In [17]:
import sys
sys.path.append('..')

In [18]:
import os

import pandas as pd

from models.load_data import load_eeg_train_data, load_eeg_test_data
from models.FB_MLP import calc_class_weights
from config.paths import PATHS
from utils.io import pickle_path

# make it only use GPU 0
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPool2D, Dropout, Flatten, Dense, \
    BatchNormalization, LeakyReLU

# Adapting the model to my thesis
Eberlein's paper's model uses a 3000 x 16 input. I have a 3105 x 2 input.

In [4]:
n_samples, n_channels = 3105, 2

In [5]:
conv2d_kwargs = {
    'use_bias': False,  # no bias because we use BatchNorm afterward
    'activation': None,  # activation is applied after BatchNorm
    'kernel_regularizer': tf.keras.regularizers.l1_l2(1e-9, 1e-9),
}
LEAKY_RELU_NEGATIVE_SLOPE = 0.3


def block_after_conv(layer_idx: int, pool_size: int, pool_padding: str = 'same'):
    return [
        BatchNormalization(name=f'batch_norm{layer_idx}'),
        LeakyReLU(negative_slope=LEAKY_RELU_NEGATIVE_SLOPE, name=f'leaky_relu{layer_idx}'),
        MaxPool2D([pool_size, 1], padding=pool_padding, name=f'max_pool{layer_idx}'),
        Dropout(0.2, name=f'dropout{layer_idx}')
    ]


def conv_block1(layer_idx: int, kernel_size: int, n_filters: int, pool_size: int):
    return [
        Conv2D(n_filters, [kernel_size, 1], padding='same', **conv2d_kwargs, name=f'conv{layer_idx}'),
        *block_after_conv(layer_idx, pool_size)
    ]


def conv_block2(layer_idx: int, kernel_size: int, n_filters1: int, n_filters2: int, pool_size: int,
                pool_padding: str = 'same'):
    return [
        Conv2D(n_filters1, [kernel_size, 1], padding='valid', **conv2d_kwargs, name=f'conv{layer_idx}.1'),
        Conv2D(n_filters2, [kernel_size, 1], padding='valid', **conv2d_kwargs, name=f'conv{layer_idx}.2'),
        *block_after_conv(layer_idx, pool_size, pool_padding)
    ]

In [6]:
model = tf.keras.models.Sequential([
    Input([n_samples, n_channels, 1]),
    BatchNormalization(name='batch_norm0'),

    *conv_block1(1, kernel_size=5, n_filters=32, pool_size=5),
    *conv_block1(2, kernel_size=5, n_filters=64, pool_size=3),
    *conv_block1(3, kernel_size=3, n_filters=96, pool_size=2),
    *conv_block1(4, kernel_size=3, n_filters=128, pool_size=2),

    *conv_block2(5, kernel_size=4, n_filters1=128, n_filters2=96, pool_size=2),
    *conv_block2(6, kernel_size=4, n_filters1=64, n_filters2=32, pool_size=2, pool_padding='valid'),
    # <- !!! changed padding to valid !!!
    *conv_block2(7, kernel_size=4, n_filters1=32, n_filters2=32, pool_size=2),

    Flatten(name='flatten8'),
    Dropout(0.5, name='dropout8'),

    # !!! changed to 8 nodes, instead of 64, because #nodes gets reduced by a factor of 8
    Dense(8, activation=None, name='dense9'),
    LeakyReLU(negative_slope=LEAKY_RELU_NEGATIVE_SLOPE, name='leaky_relu9'),

    Dense(1, activation='sigmoid', name='output')
], name='CNN',
)

I0000 00:00:1768235512.519458  938544 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22377 MB memory:  -> device: 0, name: NVIDIA RTX A5000, pci bus id: 0000:31:00.0, compute capability: 8.6


In [7]:
print(f'#layers: {len(model.layers)}')
model.summary()

#layers: 44


Model: "CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ batch_norm0                     │ (None, 3105, 2, 1)     │             4 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1 (Conv2D)                  │ (None, 3105, 2, 32)    │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_norm1                     │ (None, 3105, 2, 32)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_relu1 (LeakyReLU)         │ (None, 3105, 2, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pool1 (MaxPooling2D)        │ (None, 621, 2, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout1 (Dropout)              │ (None, 621, 2, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv2D)                  │ (None, 621, 2, 64)     │        10,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_norm2                     │ (None, 621, 2, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_relu2 (LeakyReLU)         │ (None, 621, 2, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pool2 (MaxPooling2D)        │ (None, 207, 2, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout2 (Dropout)              │ (None, 207, 2, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3 (Conv2D)                  │ (None, 207, 2, 96)     │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_norm3                     │ (None, 207, 2, 96)     │           384 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_relu3 (LeakyReLU)         │ (None, 207, 2, 96)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pool3 (MaxPooling2D)        │ (None, 104, 2, 96)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout3 (Dropout)              │ (None, 104, 2, 96)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv4 (Conv2D)                  │ (None, 104, 2, 128)    │        36,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_norm4                     │ (None, 104, 2, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_relu4 (LeakyReLU)         │ (None, 104, 2, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pool4 (MaxPooling2D)        │ (None, 52, 2, 128)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout4 (Dropout)              │ (None, 52, 2, 128)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv5.1 (Conv2D)                │ (None, 49, 2, 128)     │        65,536 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 223,797 (874.21 KB)

 Trainable params: 222,835 (870.45 KB)

 Non-trainable params: 962 (3.76 KB)

# Compiling the model

In [8]:
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.metrics import Recall, AUC

LEARNING_RATE = 0.001
model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=BinaryCrossentropy(from_logits=False),
        metrics=["accuracy", Recall(name='recall'), AUC(name='AUC')]
)

# Training the model


In [9]:
ptnt_dir = PATHS.patient_dirs()[0]
segs = pd.read_pickle(pickle_path(ptnt_dir.segments_table))
esegs = segs[segs['exists']]
split_idx = pd.read_pickle(pickle_path(ptnt_dir.train_test_split)).segment_index

In [10]:
x_train, y_train = load_eeg_train_data(esegs, split_idx, ptnt_dir.edf_dir)

In [11]:
print(x_train.shape)
print(y_train.shape)

(30051, 3105, 2)
(30051,)


In [19]:
EPOCHS = 50
BATCH_SIZE = 256  # larger batch size, so that preictal samples are most likely in every batch
class_weights = calc_class_weights(y_train)
model.fit(x_train, y_train, epochs=2, batch_size=BATCH_SIZE, class_weight=class_weights)

Epoch 1/2
118/118 ━━━━━━━━━━━━━━━━━━━━ 32s 164ms/step - AUC: 0.6184 - accuracy: 0.7087 - loss: 0.7497 - recall: 0.3767
Epoch 2/2
118/118 ━━━━━━━━━━━━━━━━━━━━ 4s 31ms/step - AUC: 0.6692 - accuracy: 0.5192 - loss: 0.6218 - recall: 0.7617


In [13]:
x_test, y_test = load_eeg_test_data(esegs, split_idx, ptnt_dir.edf_dir)

/data/home/webb/SeizurePredictionThesis/models/../models/load_data.py:140: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_segs.drop(columns=list(Features.ORDERED_NAMES), inplace=True)


In [14]:
print(x_test.shape)
print(y_test.shape)

(281691, 3105, 2)
(281691,)


In [15]:
model.evaluate(x_test, y_test)

8803/8803 ━━━━━━━━━━━━━━━━━━━━ 82s 9ms/step - AUC: 0.4481 - accuracy: 0.9949 - loss: 0.1319 - recall: 0.0000e+00


[0.13190974295139313, 0.9948915839195251, 0.0, 0.44805464148521423]